# Lab 2 - The Prompt Lab (Track N: notebook)

One file, top to bottom. Work the cells in order, run each one, read its
output. Nothing here asks you to open another script.

This repo has two tracks. **This notebook is Track N.** The alternative is
Track S - the scripts `grid.py`, `variants.py`, `stats.py` under
`labs/lab2/`. Pick one and ignore the other for the whole lab.

| Part | What you do | ~time |
|---|---|---|
| 0 | Environment + read `aip/evals.py` | 10 min |
| A | Few-shot selection | 40 min |
| B | Run the grid, build the table | 40 min |
| C | The cascade | 30 min |
| D | Confidence intervals + paired tests | 20 min |
| E | Error analysis + recommendation | 25 min |
| Submit | Copy into `variants.py` + fill `report.md` | 5 min |

**Prerequisite:** Lab 1 must be finished. This notebook imports your
`labs/lab1/extract.py`.

**The one rule:** run cells in order, and when a cell says *your turn*, edit it
there. When you are done, copy your configurations and the cascade into
`labs/lab2/variants.py` - that is the graded file - and fill `report.md`
by hand. Never regenerate the report.

## 0 - Setup

In [ ]:
# Make the shared library importable and load settings. Safe to re-run.
import os, sys, json, re
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "aip").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "aip").is_dir():
    ROOT = Path.cwd().parents[1]          # notebook opened from labs/lab2
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.chdir(ROOT)
print("repo root:", ROOT)
print("python:", sys.version.split()[0])

In [ ]:
# Dependencies + API key - the notebook's check_setup.py.
ok = True
for mod in ("pydantic", "httpx", "dotenv"):
    try:
        __import__(mod)
        print("PASS  import " + mod)
    except ImportError:
        ok = False
        print("FAIL  import " + mod + "  ->  pip install -r requirements.txt")

have_key = bool(os.getenv("GEMINI_API_KEY", "").strip())
print(("PASS" if have_key else "FAIL") + "  GEMINI_API_KEY set")
if not have_key:
    print("      create .env in the repo root with GEMINI_API_KEY=AIza...")
print("Environment is ready." if (ok and have_key) else "Fix the FAILs above.")

In [ ]:
# Lab 1 must import cleanly before anything in Lab 2 means anything.
from labs.lab1.extract import (
    SYSTEM_PROMPT, TicketRecordC, apply_business_rules,
    extract_deterministic,
)
print("lab1 extract.py imported OK")
print("TicketRecordC fields:", list(TicketRecordC.model_fields))

# If these raise NotImplementedError, finish Lab 1 first.
try:
    extract_deterministic("test AUR-1234567")
    apply_business_rules({"urgency": 3}, "test")
    print("lab1 deterministic helpers implemented")
except NotImplementedError:
    print("LAB 1 INCOMPLETE - finish extract.py Part C before Lab 2.")

In [ ]:
# Load the golden sets. dev = iterate here. test = run once, at the end.
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

dev  = load_jsonl(ROOT / "data/eval/extraction_dev.jsonl")
test = load_jsonl(ROOT / "data/eval/extraction_test.jsonl")
by_id = {r["id"]: r for r in dev}
print("dev:", len(dev), "tickets   test:", len(test), "tickets")
print("example ids:", [r["id"] for r in dev[:8]])

### Read `aip/evals.py` - all of it

It is the instrument. You are responsible for knowing what it measures:
field vs record accuracy, `run_eval`, `compare`, the report shape.

Then run `stats.py` and understand its output:

In [ ]:
%run labs/lab2/stats.py

In [ ]:
from labs.lab2.stats import wilson_interval, paired_test

# The normal approximation at n=60, p=0.90: half-width ~0.076.
# 0.88 and 0.91 sit inside each other's intervals. You cannot claim B > A.
print("normal CI for p=0.90, n=60:")
p, n = 0.90, 60
half = 1.96 * (p * (1 - p) / n) ** 0.5
print(f"  {p:.3f} +/- {half:.3f}  ->  [{p - half:.3f}, {p + half:.3f}]")
print("wilson(54/60):", wilson_interval(54, 60))
print("wilson(53/60):", wilson_interval(53, 60))

## A - Few-shot selection

Pick 6 dev tickets **by hand**. Edges, not averages. For each, one line:
*what does this teach that prose cannot?*

Then render them into the prompt in **exactly** the output format you ask for.

In [ ]:
# Browse candidates before you commit. Look for edges:
#   billing/complaint boundary - no policy number - Hinglish -
#   sentiment/urgency trap - policy only in a quoted reply - one you missed in Lab 1
for r in dev[:12]:
    e = r["expected"]
    print(r["id"],
          "|", e["category"],
          "| u" + str(e["urgency"]),
          "|", e["sentiment"],
          "|", e["language"],
          "| pn=" + str(e["policy_number"]))
    print("  ", r["input"][:140].replace("\n", " "))

In [ ]:
# YOUR TURN - A1: six ids + one line each on what it teaches.
FEW_SHOT_IDS = [
    # "Txxxx",  # teaches: ...
]

FEW_SHOT_WHY = {
    # "Txxxx": "...",
}

In [ ]:
def load_examples(ids):
    missing = [i for i in ids if i not in by_id]
    if missing:
        raise KeyError("unknown example ids: " + str(missing))
    return [by_id[i] for i in ids]

def few_shot_block(ids):
    """Render examples into the prompt. Format must match the requested
    output format byte-for-byte - a mismatch is a classic own goal."""
    if not ids:
        raise ValueError("TODO A1: fill FEW_SHOT_IDS first")
    # YOUR TURN - A2:
    #   for each example, emit:
    #     --- Example Ticket (Txxxx) ---
    #     <ticket text>
    #     Output JSON:
    #     { evidence, category, urgency, sentiment, product, language }
    #   built from ex["expected"]. Join the blocks and return them.
    raise NotImplementedError("TODO A2: implement few_shot_block()")

In [ ]:
from labs.lab2.variants import zero_shot, few_shot, few_shot_reasoned, cascade

def smoke(system, n=3, label=""):
    for r in dev[:n]:
        try:
            out = system(r["input"])
            if isinstance(out, dict):
                print(label, r["id"], out.get("category"), "u" + str(out.get("urgency")))
            else:
                print(label, r["id"], out)
        except NotImplementedError as exc:
            print(label, "not ready:", exc)
            return
        except Exception as exc:
            print(label, r["id"], type(exc).__name__ + ":", exc)

smoke(zero_shot, label="zero_shot")

In [ ]:
# A3 - compare zero-shot vs few-shot on a cheap slice first, then full dev.
# Full run (when FEW_SHOT_IDS and few_shot_block are done):
#   %run labs/lab2/grid.py --variants zero_shot few_shot --split dev

from labs.lab2.grid import metric
from aip.evals import run_eval

def run_variant(name, system, rows, budget=0.30):
    rep = run_eval(name, rows, system, metric, budget_usd=budget, workers=4)
    print(rep.summary())
    return rep

# Uncomment when your few-shot cells are implemented:
# zs = run_variant("zero_shot", zero_shot, dev, budget=0.30)
# fs = run_variant("few_shot", few_shot, dev, budget=0.30)

### A4 - the question that matters

Your 6 examples came out of the dev set, and you measured on the dev set.

**Name the problem. Fix it. Say in your report what you did.**

> **Checkpoint - prepare to be annoyed.** Few-shot may buy you *nothing*.
> Reference: field accuracy 0.928 -> 0.931, record accuracy *down*, p = 1.00.
> The zero-shot prompt already carries the labelling rules in the Pydantic
> field `description`s. If few-shot buys you 20 points, your zero-shot
> prompt was under-specified - fix that instead of banking the delta.

In [ ]:
# YOUR TURN - write your A4 answer here (it goes in report.md too).
A4_ANSWER = """
Problem:
Fix:
Honest number I will report:
""".strip()
print(A4_ANSWER)

## B - Run the grid

7 configurations: 3 prompts x 2 tiers, plus the cascade.
One row each. Quality x cost x latency - never quality alone.

```bash
python labs/lab2/grid.py --all --split dev --save reports/lab2_grid.json
```

In [ ]:
# Full grid from the notebook (same code as grid.py --all):
# %run labs/lab2/grid.py --all --split dev --save reports/lab2_grid.json

# Or piece by piece while iterating:
from labs.lab2.variants import VARIANTS
print("variants:", list(VARIANTS))

reps = {}
for name in ["zero_shot", "zero_shot_main"]:   # start cheap; add the rest
    reps[name] = run_variant(name, VARIANTS[name], dev, budget=0.40)

In [ ]:
# Build the comparison table. Paste this into report.md when the grid is done.
def grid_row(name, rep):
    agg = rep.aggregate()
    b = rep.budget
    per_ticket = b.get("cost_usd", 0) / max(rep.n, 1)
    return {
        "config": name,
        "record_accuracy": round(agg.get("record_accuracy", 0), 4),
        "field_accuracy": round(agg.get("field_accuracy", 0), 4),
        "schema_valid": round(agg.get("schema_valid", 0), 4),
        "cost_usd": round(b.get("cost_usd", 0), 4),
        "cost_per_1k": round(per_ticket * 1000, 2),
        "p50_ms": round(b.get("latency_p50_ms", 0)),
        "p95_ms": round(b.get("latency_p95_ms", 0)),
        "escalated": round(agg.get("escalated", 0), 4),
    }

rows = [grid_row(n, r) for n, r in reps.items()]
cols = list(rows[0]) if rows else []
print(" | ".join(cols))
print("-|-" * len(cols))
for row in rows:
    print(" | ".join(str(row[c]) for c in cols))

# YOUR TURN - answer in report.md:
# 1. Which axis moved the numbers most - prompt, or model tier?
# 2. Reasoning field: accuracy points per rupee (or per 1k tokens).
# 3. Any configuration dominated on every axis? Name it.

## C - The cascade

```
   SMALL
     |- validates AND evidence non-empty -> accept
     |- otherwise -> MAIN -> accept, or flag for review
```

Report **escalation rate**, **blended cost**, **blended accuracy**.

> **The silent bug.** Two samples of the *same* request at temperature 0
> hit the cache; answers are byte-identical; escalation stays 0%; nothing
> errors. Second sample must use temperature > 0.
>
> **Then check the trigger has signal:** agreement when correct vs when
> wrong. Reference: 94% vs 83% - caught 2 of 12 errors. Disagreement
> detects variance; the model has bias.

In [ ]:
from aip.llm import structured, StructuredOutputError
from labs.lab1.extract import SYSTEM_PROMPT, TicketRecordC

def cascade_notebook(ticket):
    """YOUR TURN - C: SMALL first; escalate to MAIN on your trigger.

    Set rec["_path"] = "small" | "large" so the grid can report
    the escalation rate.
    """
    det = extract_deterministic(ticket)
    try:
        rec1 = structured(ticket, schema=TicketRecordC,
                          system=SYSTEM_PROMPT, tier="SMALL")
        m = rec1.model_dump()
        ev = m.get("evidence", "").strip()
    except Exception:
        return _main(ticket, det)

    # TODO C: choose your trigger(s).
    #   if len(ev) < 5: return _main(ticket, det)
    #   second = structured(..., tier="SMALL", temperature=0.7)  # T > 0!
    #   if category disagrees: return _main(ticket, det)

    m["needs_human_review"] = False
    m["review_reason"] = ""
    m["_path"] = "small"
    return apply_business_rules({**m, **det}, ticket)

def _main(ticket, det):
    try:
        rec = structured(ticket, schema=TicketRecordC,
                         system=SYSTEM_PROMPT, tier="MAIN")
        m = rec.model_dump()
        m["needs_human_review"] = False
        m["review_reason"] = ""
    except (StructuredOutputError, Exception) as exc:
        m = {"evidence": "", "category": "information", "urgency": 1,
             "sentiment": "neutral", "product": "unknown", "language": "en",
             "needs_human_review": True,
             "review_reason": type(exc).__name__ + ": " + str(exc)}
    m["_path"] = "large"
    return apply_business_rules({**m, **det}, ticket)

print("cascade_notebook defined - implement the trigger, then run:")
print("  run_variant(\"cascade\", cascade_notebook, dev)")

In [ ]:
# Run the cascade and report the three numbers.
# casc = run_variant("cascade", cascade_notebook, dev, budget=0.40)
# agg = casc.aggregate()
# print("escalation rate:", agg.get("escalated", 0))
# print("blended cost:    $", casc.budget.get("cost_usd", 0))
# print("blended record:  ", agg.get("record_accuracy", 0))
# print("blended field:   ", agg.get("field_accuracy", 0))

In [ ]:
# Signal check: agreement when correct vs when wrong (needs cascade + gold).
# YOUR TURN - after the cascade runs:
# agree_correct = disagree_correct = 0
# agree_wrong = disagree_wrong = 0
# ... compare two T>0 samples against gold ...
# print("agree when correct:", ...
# print("agree when wrong:  ", ...
print("(fill after cascade is implemented)")

## D - Is your difference real?

1. Wilson CI for each configuration (`wilson_interval`).
2. Paired McNemar on the same items (`paired_test`): b, c, p.
3. **"No significant difference" is a good result** - choose on cost.

In [ ]:
def correct_map(rep):
    return {r.id: r.metrics.get("record_accuracy", 0) == 1.0
            for r in rep.results}

def compare_pair(rep_a, rep_b, label_a="A", label_b="B"):
    a, b = correct_map(rep_a), correct_map(rep_b)
    ids = [i for i in a if i in b]
    res = paired_test([a[i] for i in ids], [b[i] for i in ids])
    print(f"{label_a} vs {label_b}: b={res['b']} c={res['c']} "
          f"p={res['p_value']:.4f}")
    print("  ", res["verdict"])
    ca = sum(a[i] for i in ids)
    cb = sum(b[i] for i in ids)
    print(f"   CI {label_a}: {wilson_interval(ca, len(ids))}")
    print(f"   CI {label_b}: {wilson_interval(cb, len(ids))}")
    return res

# When you have two reports:
# compare_pair(reps["zero_shot"], reps["zero_shot_main"],
#              "zero_shot", "zero_shot_main")
print("run after you have at least two reports in `reps`")

In [ ]:
# YOUR TURN - D3: your chosen comparison, in words for report.md:
D3_ANSWER = """
comparison:
b =   c =   p =
conclusion:
""".strip()
print(D3_ANSWER)

## E - Error analysis and the recommendation

1. Read 20 failures. Cluster them. Top three with counts.
2. Confusion matrix on your worst field - find the systematic confusion.
3. One paragraph: configuration, three numbers, annual cost at 10k/day,
   one change-my-mind condition.

In [ ]:
# Failures from your best configuration - actually read them.
def show_failures(rep, limit=20):
    for r in rep.failures("record_accuracy", limit=limit):
        wrong = [k.removeprefix("field.") for k, v in r.metrics.items()
                 if k.startswith("field.") and v == 0.0]
        print(r.id, "wrong=" + str(wrong), r.error or "")

# show_failures(reps["zero_shot_main"])

In [ ]:
from collections import Counter

def confusion(rep, cases, field):
    gold = {c.id: (c.expected or {}).get(field) for c in cases}
    # cases are aip.evals.Case when coming from run_eval; rebuild from jsonl:
    gold = {r["id"]: r["expected"].get(field) for r in dev}
    counts = Counter()
    for r in rep.results:
        pred = r.output.model_dump() if hasattr(r.output, "model_dump") else r.output
        p = (pred or {}).get(field) if isinstance(pred, dict) else None
        counts[(gold.get(r.id), p)] += 1
    labels = sorted({v for pair in counts for v in pair if v is not None}, key=str)
    w = max((len(str(v)) for v in labels), default=8) + 2
    print("confusion for", field, " (rows=gold, cols=pred)")
    print(" " * w + "".join(f"{str(v):>{w}}" for v in labels))
    for g in labels:
        row = f"{str(g):<{w}}"
        for p in labels:
            row += f"{(counts.get((g, p), 0) or '.'):>{w}}"
        print(row)

# confusion(reps["zero_shot_main"], dev, "urgency")

In [ ]:
# YOUR TURN - E1 clusters, E2 finding, E3 recommendation (report.md).
E1_CLUSTERS = [
    # (name, count, fix),
]

E3_RECOMMENDATION = """
Ship <configuration>. <record> record / <field> field / $<cost> per 1k /
<p95> ms p95. Annual cost at 10k/day: $<year>. Change my mind if: <condition>.
""".strip()

print(E3_RECOMMENDATION)

In [ ]:
# Calibration (reference solution, test split):
#   SMALL zero-shot: 0.930 field / $0.080 / 2,276 ms p95
#   MAIN  zero-shot: 0.942 field / $0.382 / 6,720 ms p95
#   4.8x cost, 3x latency, paired p = 0.71 (not distinguishable from noise).
# "Ship the cheap one" is a complete and correct recommendation.

TICKETS_PER_DAY = 10_000
def annual_cost(per_ticket_usd):
    return per_ticket_usd * TICKETS_PER_DAY * 365

# print(annual_cost(<per-ticket cost from your best row>))

## Negative results (required - at least one)

Few-shot bought nothing? Reasoning field cost more than it bought? Cascade
escalates 100%? No configuration beat the cheap baseline? **Write it down.**
That is full marks, not a penalty.

In [ ]:
NEGATIVE_RESULTS = [
    # "few-shot: p=..., no detectable gain - descriptions already carry the rules",
]
for i, line in enumerate(NEGATIVE_RESULTS, 1):
    print(i, line)

## Submit - turn the notebook into the graded files

You built everything here. When you are done:

1. Copy your `FEW_SHOT_IDS`, `few_shot_block`, and `cascade` into
   `labs/lab2/variants.py` (the only file graded).
2. Fill in `labs/lab2/report.md` yourself. Never regenerate it - the tables
   are yours to complete from the harness output.
3. Save the grid: `reports/lab2_grid.json`
   (`%run labs/lab2/grid.py --all --split dev --save reports/lab2_grid.json`).

Report must contain: grid table · A1 + A4 · cascade three numbers ·
paired test (b, c, p) · three clusters + confusion matrix · recommendation ·
**at least one negative result**.

### Cheat sheet

- `wilson_interval(successes, n)` / `paired_test(a_correct, b_correct)` - from
  `labs/lab2/stats.py`. Neither is optional.
- `run_eval(name, cases, system, metric, budget_usd=..., workers=...)` -
  from `aip.evals`. Same harness as `grid.py`.
- Second sample for disagreement: `temperature=0.7` (temperature 0 = cache hit).
- Cascade path: `rec["_path"] = "small" | "large"`.
- No accuracy target. Grade = quality of the decision + the evidence.
- Brief and rubric: `labs/lab2/README.md`.
- Procedure and commands: `labs/lab2/RUNSHEET.md`.
- Theory and full war stories: `labs/lab2/CONCEPTS.md`.